# Kiểm tra dữ liệu ban đầu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Initial data check

In [ ]:
#1. Load dataset
df = pd.read_csv("../../data/raw/phone/phone_specs.csv")
df.head()

In [ ]:
# Kiểm tra kích thước và danh sách cột
print("KÍCH THƯỚC DỮ LIỆU")
print("-" * 50)
print(f"Số dòng: {df.shape[0]}")
print(f"Số cột : {df.shape[1]}")

print("\nDANH SÁCH CỘT")
print("-" * 50)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
# Check tổng hợp dữ liệu ban đầu

print("=" * 80)
print("1. THÔNG TIN KIỂU DỮ LIỆU, NULL, UNIQUE")
print("=" * 80)

overview_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_percent": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

display(overview_df)


print("\n" + "=" * 80)
print("2. KIỂM TRA MISSING VALUES")
print("=" * 80)

missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values
}).sort_values(by="missing_count", ascending=False)

display(missing_df)


print("\n" + "=" * 80)
print("3. KIỂM TRA DỮ LIỆU TRÙNG LẶP")
print("=" * 80)

print(f"Số dòng trùng toàn bộ: {df.duplicated().sum()}")

if "Tên sản phẩm" in df.columns:
    print(f"Số sản phẩm trùng tên: {df.duplicated(subset=['Tên sản phẩm']).sum()}")

if "URL" in df.columns:
    print(f"Số URL trùng: {df.duplicated(subset=['URL']).sum()}")


print("\n" + "=" * 80)
print("4. HIỂN THỊ CÁC SẢN PHẨM BỊ TRÙNG TÊN")
print("=" * 80)

if "Tên sản phẩm" in df.columns:
    duplicated_products = df[df.duplicated(subset=["Tên sản phẩm"], keep=False)]

    if duplicated_products.empty:
        print("Không có sản phẩm trùng tên.")
    else:
        cols_to_show = [
            col for col in ["Tên sản phẩm", "Giá bán", "RAM", "Bộ nhớ trong", "URL"]
            if col in df.columns
        ]
        display(
            duplicated_products[cols_to_show]
            .sort_values(by="Tên sản phẩm")
        )


print("\n" + "=" * 80)
print("5. SAMPLE GIÁ TRỊ CỦA TỪNG CỘT")
print("=" * 80)

sample_values = []

for col in df.columns:
    values = df[col].dropna().astype(str).unique()[:5]
    sample_values.append({
        "column": col,
        "sample_values": list(values)
    })

sample_df = pd.DataFrame(sample_values)

display(sample_df)


# 2. Standardize Column Names


In [ ]:
# Define column renaming dictionary
rename_map = {
    "Tên sản phẩm": "product_name",
    "Giá bán": "price_raw",
    "RAM": "ram_raw",
    "Bộ nhớ trong": "storage_raw",
    "Kích thước màn hình": "screen_size_raw",
    "Tấm nền": "panel_raw",
    "Tần số quét": "refresh_rate_raw",
    "Độ phân giải": "resolution_raw",
    "Dung lượng pin": "battery_raw",
    "Chip xử lý": "chip_raw",
    "Hãng sản xuất": "brand_raw",
    "Hệ điều hành": "os_raw",
    "Camera trước": "front_camera_raw",
    "Camera sau": "rear_camera_raw",
    "Quay video": "video_raw",
    "Sạc nhanh": "charging_raw",
    "Kháng nước / bụi": "ip_raw",
    "Trọng lượng": "weight_raw",
    "Khuyến mãi giảm giá": "promotion_raw",
    "URL": "url"
}

print("Column renaming dictionary created successfully.")

In [ ]:
# Rename columns
df = df.rename(columns=rename_map)

print("Columns renamed successfully.")

In [ ]:
# Check renamed columns
print("UPDATED COLUMN NAMES")
print("-" * 50)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print('\nSample new dataframe')
display(df.head(5))

# 3. Handle Duplicate Products

In [ ]:
# Check duplicate product names
duplicate_name_count = df.duplicated(subset=["product_name"]).sum()

print("Duplicate product name count:", duplicate_name_count)

if duplicate_name_count > 0:
    duplicate_names = df[df.duplicated(subset=["product_name"], keep=False)]

    columns_to_show = [
        "product_name",
        "price_raw",
        "ram_raw",
        "storage_raw",
        "url"
    ]

    display(
        duplicate_names[columns_to_show]
        .sort_values(by="product_name")
    )
else:
    print("No duplicate product names found")

In [ ]:
# Remove duplicates
df = df.drop_duplicates(subset=["product_name"], keep="first").reset_index(drop=True)

print("Duplicate product_name removed successfully.")
print("Shape after removing duplicates:", df.shape)

# 4. Parse Simple Raw Product Columns

In [ ]:
# Bookmark 4.2 — Define helper parsing functions

import re

def clean_text(value):
    """
    Clean raw text by removing leading/trailing spaces
    and reducing multiple spaces into one space.

    This function does not transform the meaning of the text.
    It only makes text columns cleaner.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)

    return text


def clean_price(price):
    """
    Parse product price from raw text into integer VND.

    Example:
    '5.790.000đ' -> 5790000
    '530.000đ'   -> 530000

    Note:
    This function only handles product price in the dataset.
    User budget input will be handled later in the recommendation system.
    """
    if pd.isna(price):
        return np.nan

    text = str(price)
    text = text.replace("đ", "")
    text = text.replace("₫", "")
    text = text.replace(".", "")
    text = text.replace(",", "")
    text = text.replace(" ", "")

    digits = re.findall(r"\d+", text)

    if len(digits) == 0:
        return np.nan

    return int("".join(digits))


def parse_memory_to_gb(value):
    """
    Parse memory value into GB.

    Examples:
    '8 GB'     -> 8
    '128 GB'   -> 128
    '1 TB'     -> 1024
    '128 MB'   -> 0.125

    Important:
    For RAM values like '8GB + Mở rộng 10GB',
    only the first memory value is parsed as physical RAM.

    Virtual RAM is treated as an additional feature,
    so it will not be extracted in this preprocessing step.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    text = text.replace(",", ".")

    matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*(TB|GB|MB)\b",
        text,
        flags=re.IGNORECASE
    )

    if len(matches) == 0:
        return np.nan

    # Use only the first memory value.
    # This avoids mixing physical RAM with virtual RAM.
    number = float(matches[0][0])
    unit = matches[0][1].upper()

    if unit == "TB":
        return number * 1024
    elif unit == "GB":
        return number
    elif unit == "MB":
        return number / 1024

    return np.nan


def parse_screen_size(value):
    """
    Parse screen size into inches.

    Example:
    '6.7 inches'  -> 6.7
    '6,7 inches'  -> 6.7
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    text = text.replace(",", ".")

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*(inch|inches)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    # Fallback: extract the first decimal-like number
    match = re.search(r"\d+(?:\.\d+)?", text)

    if match:
        return float(match.group(0))

    return np.nan


def clean_panel(value):
    """
    Clean display panel text only.

    This function does not group panel types.
    Panel grouping such as AMOLED, OLED, LCD, IPS LCD
    will be handled later in feature engineering.
    """
    return clean_text(value)


def parse_refresh_rate(value):
    """
    Parse refresh rate into Hz.

    Examples:
    '120Hz'              -> 120
    '120 Hz'             -> 120
    'Từ 144Hz trở lên'   -> 144
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    matches = re.findall(
        r"(\d+)\s*Hz",
        text,
        flags=re.IGNORECASE
    )

    if len(matches) == 0:
        return np.nan

    # If multiple refresh rates appear, keep the highest one.
    return int(max([int(x) for x in matches]))


def parse_resolution(value):
    """
    Parse screen resolution into width, height, and total pixels.

    Examples:
    '1080 x 2340 pixels' -> 1080, 2340, 2527200
    '1608 × 720 pixels'  -> 720, 1608, 1157760
    'QVGA 240*320'       -> 240, 320, 76800

    Note:
    This function only extracts numeric resolution values.
    Resolution grouping will be handled later.
    """
    if pd.isna(value):
        return np.nan, np.nan, np.nan

    text = str(value).strip()

    match = re.search(
        r"(\d{3,4})\s*[x×*]\s*(\d{3,4})",
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return np.nan, np.nan, np.nan

    number_1 = int(match.group(1))
    number_2 = int(match.group(2))

    # Use short side and long side because phone resolution orientation
    # may be written inconsistently across products.
    resolution_width = min(number_1, number_2)
    resolution_height = max(number_1, number_2)
    resolution_total_pixels = resolution_width * resolution_height

    return resolution_width, resolution_height, resolution_total_pixels


def parse_battery(value):
    """
    Parse battery capacity into mAh.

    Examples:
    '6000 mAh'  -> 6000
    '2300mAh'   -> 2300
    '3,095mAh'  -> 3095  (comma as thousand separator, not decimal)
    '5,000mAh'  -> 5000
    '4,400 mAh' -> 4400

    Fix:
    Commas in battery values are thousand separators (3,095 = 3095).
    Remove commas before parsing to avoid incorrect splits.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Remove thousand separator commas before parsing
    # Example: "3,095mAh" → "3095mAh", "5,000mAh" → "5000mAh"
    text = re.sub(r"(\d),(\d)", r"\1\2", text)

    matches = re.findall(
        r"(\d{3,5})\s*m\s*ah",
        text,
        flags=re.IGNORECASE
    )

    if len(matches) > 0:
        return int(matches[0])

    # Fallback: extract a 3 to 5 digit number
    match = re.search(r"\d{3,5}", text)

    if match:
        return int(match.group(0))

    return np.nan


def parse_fast_charge(value):
    """
    Parse charging power into watts.

    Examples:
    'Sạc nhanh 44W'       -> 44
    'Sạc siêu nhanh 45 W' -> 45
    'Fast charging 15W'   -> 15

    Note:
    This only extracts charging wattage.
    Fast charging classification will be handled later.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*W\b",
        text,
        flags=re.IGNORECASE
    )

    if len(matches) == 0:
        return np.nan

    # If multiple watt values appear, keep the highest one.
    return float(max([float(x) for x in matches]))


def parse_weight(value):
    """
    Parse phone weight into grams.

    Examples:
    '199g'  -> 199
    '195 g' -> 195
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    text = text.replace(",", ".")

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*gr?\b",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return np.nan


def standardize_brand(value):
    """
    Standardize brand names.

    This function only fixes common casing and naming format.
    Brand scoring or brand grouping will be handled later.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    upper_text = text.upper()

    brand_map = {
        "APPLE": "Apple",
        "SAMSUNG": "Samsung",
        "XIAOMI": "Xiaomi",
        "OPPO": "OPPO",
        "VIVO": "Vivo",
        "REALME": "Realme",
        "NOKIA": "Nokia",
        "HONOR": "Honor",
        "TECNO": "Tecno",
        "NUBIA": "Nubia",
        "ASUS": "ASUS",
        "ZTE": "ZTE",
        "SONY": "Sony",
        "NOTHING": "Nothing",
        "MASSTEL": "Masstel",
        "ITEL": "itel",
        "MEIZU": "Meizu",
        "VSMART": "Vsmart",
        "INFINIX": "Infinix",
        "BENCO": "Benco"
    }

    return brand_map.get(upper_text, text.title())


def parse_os_family(value):
    """
    Parse operating system family from raw OS text.

    Examples:
    'Android 14, Funtouch OS 14' -> Android
    'iOS 18'                     -> iOS

    Note:
    This only extracts OS family.
    OS scoring will be handled later.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).lower()

    if "ios" in text:
        return "iOS"
    elif "android" in text:
        return "Android"
    elif "harmony" in text:
        return "HarmonyOS"
    else:
        return "Other"


def parse_os_version(value):
    """
    Parse OS version number.

    Examples:
    'Android 14' -> 14
    'iOS 18'     -> 18
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    match = re.search(
        r"(Android|iOS)\s*(\d+(?:\.\d+)?)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return float(match.group(2))

    return np.nan


def parse_ip_rating(value):
    """
    Parse IP rating from raw waterproof/dustproof text.

    Examples:
    'IP68'                -> IP68
    'Đạt mức IP68...'     -> IP68
    'Không'               -> none

    Note:
    This only extracts the IP rating.
    Water resistance scoring will be handled later.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).upper()

    match = re.search(
        r"IP(?:\d{2}|X\d|\dX)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).upper()

    if "KHÔNG" in text or "KHONG" in text:
        return "none"

    return np.nan


def clean_chip_name(value):
    """
    Clean chip name text only.

    Examples:
    'Helio G85 (12nm)'       -> 'Helio G85 (12nm)'
    'Snapdragon 6 Gen 3'     -> 'Snapdragon 6 Gen 3'

    Note:
    Chip brand, chip tier, benchmark score, and performance score
    require deeper processing, so they will be handled later.
    """
    return clean_text(value)


def clean_promotion_text(value):
    """
    Clean promotion text only.

    This step does not extract voucher value or promotion type.
    Promotion parsing is more context-dependent and can be handled later.
    """
    return clean_text(value)


print("Helper parsing functions for simple raw product columns defined successfully.")

In [ ]:
# Apply basic text cleaning

# Clean product name
df["product_name_clean"] = df["product_name"].apply(clean_text)

# Clean promotion text only.
# Promotion extraction will be handled later if needed.
df["promotion_clean"] = df["promotion_raw"].apply(clean_promotion_text)

print("Basic text columns cleaned successfully.")

# Display raw columns and newly created columns
display(
    df[
        [
            "product_name",
            "product_name_clean",
            "promotion_raw",
            "promotion_clean"
        ]
    ].head(10)
)

In [ ]:
# Apply price and memory parsing

# Parse product price into numeric VND
df["price_vnd"] = df["price_raw"].apply(clean_price)

# Parse physical RAM only.
# Example: '8GB + Mở rộng 10GB' -> ram_gb = 8
# Virtual RAM is ignored in this step because it is only an additional feature.
df["ram_gb"] = df["ram_raw"].apply(parse_memory_to_gb)

# Parse internal storage into GB
df["storage_gb"] = df["storage_raw"].apply(parse_memory_to_gb)

print("Price and memory columns parsed successfully.")

# Display raw columns and newly created columns
display(
    df[
        [
            "product_name",
            "price_raw",
            "price_vnd",
            "ram_raw",
            "ram_gb",
            "storage_raw",
            "storage_gb"
        ]
    ].head(15)
)

In [ ]:
# Apply display parsing

# Parse screen size into inches
df["screen_size_inch"] = df["screen_size_raw"].apply(parse_screen_size)

# Clean panel text only.
# Panel grouping will be handled later.
df["panel_clean"] = df["panel_raw"].apply(clean_panel)

# Parse refresh rate into Hz
df["refresh_rate_hz"] = df["refresh_rate_raw"].apply(parse_refresh_rate)

# Parse screen resolution into width, height, and total pixels
resolution_parsed = df["resolution_raw"].apply(
    lambda x: pd.Series(
        parse_resolution(x),
        index=[
            "resolution_width",
            "resolution_height",
            "resolution_total_pixels"
        ]
    )
)

df[
    [
        "resolution_width",
        "resolution_height",
        "resolution_total_pixels"
    ]
] = resolution_parsed

print("Display columns parsed successfully.")

# Display raw columns and newly created columns
display(
    df[
        [
            "product_name",
            "screen_size_raw",
            "screen_size_inch",
            "panel_raw",
            "panel_clean",
            "refresh_rate_raw",
            "refresh_rate_hz",
            "resolution_raw",
            "resolution_width",
            "resolution_height",
            "resolution_total_pixels"
        ]
    ].head(15)
)

In [ ]:
# Apply battery, charging, and weight parsing

# Parse battery capacity into mAh
df["battery_mah"] = df["battery_raw"].apply(parse_battery)

# Parse charging power into watts
df["fast_charge_w"] = df["charging_raw"].apply(parse_fast_charge)

# Parse phone weight into grams
df["weight_g"] = df["weight_raw"].apply(parse_weight)

print("Battery, charging, and weight columns parsed successfully.")

# Display raw columns and newly created columns
display(
    df[
        [
            "product_name",
            "battery_raw",
            "battery_mah",
            "charging_raw",
            "fast_charge_w",
            "weight_raw",
            "weight_g"
        ]
    ].head(15)
)

In [ ]:
# Apply brand, OS, IP rating, and chip parsing

# Standardize brand name
df["brand"] = df["brand_raw"].apply(standardize_brand)

# Parse operating system family
df["os_family"] = df["os_raw"].apply(parse_os_family)

# Parse operating system version
df["os_version"] = df["os_raw"].apply(parse_os_version)

# Parse IP rating
df["ip_rating"] = df["ip_raw"].apply(parse_ip_rating)

# Clean chip name only.
# Chip brand, chip tier, and chip score will be handled later.
df["chip_name_clean"] = df["chip_raw"].apply(clean_chip_name)

print("Brand, OS, IP rating, and chip columns parsed successfully.")

# Display raw columns and newly created columns
display(
    df[
        [
            "product_name",
            "brand_raw",
            "brand",
            "os_raw",
            "os_family",
            "os_version",
            "ip_raw",
            "ip_rating",
            "chip_raw",
            "chip_name_clean"
        ]
    ].head(15)
)

In [ ]:
# Final check for all parsed columns

b4_numeric_cols = [
    "price_vnd",
    "ram_gb",
    "storage_gb",
    "screen_size_inch",
    "refresh_rate_hz",
    "resolution_width",
    "resolution_height",
    "resolution_total_pixels",
    "battery_mah",
    "fast_charge_w",
    "weight_g",
]

b4_text_cols = [
    "product_name_clean",
    "promotion_clean",
    "panel_clean",
    "brand",
    "os_family",
    "os_version",
    "ip_rating",
    "chip_name_clean",
]

print("=" * 55)
print("BOOKMARK 4 — SIMPLE PARSED COLUMNS SUMMARY")
print("=" * 55)

print("\n--- Numeric columns ---")
for col in b4_numeric_cols:
    n_parsed  = df[col].notna().sum()
    n_missing = df[col].isna().sum()
    print(f"\n[ {col} ]")
    print(f"  Parsed  : {n_parsed}")
    print(f"  Missing : {n_missing}")
    print(f"  Min     : {df[col].min()}")
    print(f"  Max     : {df[col].max()}")

print("\n--- Text columns ---")
for col in b4_text_cols:
    n_parsed  = df[col].notna().sum()
    n_missing = df[col].isna().sum()
    n_unique  = df[col].nunique()
    print(f"\n[ {col} ]")
    print(f"  Parsed  : {n_parsed}")
    print(f"  Missing : {n_missing}")
    print(f"  Unique  : {n_unique}")

# 5. Parse Camera and Video Columns


In [ ]:
# Define helper parsing functions for camera and video columns

import re

def parse_front_camera_mp(value):
    """
    Parse front camera MP from raw text.
    Only extract the first MP value found.

    Examples:
    '13MP, f/2.0'                        -> 13.0
    '12 MP, f/2.2, 23mm (wide)'          -> 12.0
    '10MP (bên ngoài) + 4MP (dưới màn)' -> 10.0
    'Camera trước: 32MP, f/2.0'          -> 32.0
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    match = re.search(r"(\d+(?:\.\d+)?)\s*MP", text, flags=re.IGNORECASE)
    if match:
        return float(match.group(1))

    return np.nan


def parse_rear_main_camera_mp(value):
    """
    Parse the first rear camera MP value, which is usually the main lens.

    The first MP value found is used because the main lens is almost always
    listed first. Taking the largest MP would be incorrect — for example:
        'Camera chính: 12 MP ... Camera tele: 64 MP'
    The main lens is 12 MP, not 64 MP.

    Examples:
    'Chính 50 MP, f/1.8 Phụ 2 MP, f/2.4'                -> 50.0
    '50.0 MP, F/1.8 + 8.0 MP, F/2.2 + 5.0 MP, F/2.4'   -> 50.0
    'Camera chính: 12 MP, f/1.8 ... Camera tele: 64 MP'  -> 12.0
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    match = re.search(r"(\d+(?:\.\d+)?)\s*MP", text, flags=re.IGNORECASE)
    if match:
        return float(match.group(1))

    return np.nan


def parse_rear_camera_count(value):
    """
    Count the number of rear camera lenses by counting MP mentions.

    Examples:
    'Chính 50 MP, f/1.8 Phụ 2 MP, f/2.4'                     -> 2
    '50.0 MP, F/1.8 + 8.0 MP, F/2.2 + 5.0 MP, F/2.4'        -> 3
    'Camera chính: 12 MP ... Camera tele: 64 MP ... 12 MP ...' -> 3
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    matches = re.findall(r"\d+(?:\.\d+)?\s*MP", text, flags=re.IGNORECASE)

    if len(matches) == 0:
        return np.nan

    return int(len(matches))


# Resolution priority order — checked from highest to lowest
# Labels are kept as-is from the data, not normalized to "4K"
RESOLUTION_PRIORITY = [
    (r"\b8K\b",                 "8K"),
    (r"\b4320p\b",              "4320p"),
    (r"\b3240p\b",              "3240p"),
    (r"\b(4K|UHD|2160p)\b",    "4K"),
    (r"\b(2K|1440p)\b",        "2K"),
    (r"\b(1080p|FullHD|FHD)\b","1080p"),
    (r"\b(720p|HD)\b",         "720p"),
]


def parse_max_video_resolution(value):
    """
    Parse the maximum video resolution from raw text.
    Resolution priority (highest to lowest):
        8K > 4320p > 3240p > 4K/UHD/2160p > 2K/1440p > 1080p/FullHD > 720p/HD

    Examples:
    'UHD 4K (3840 x 2160)@30fps, 240fps @HD'  -> '4K'
    '3240p@30fps, 2160p@30/60fps'              -> '3240p'
    'HD 720p@30fps, FullHD 1080p@30fps'        -> '1080p'
    'UHD 8K (7680 x 4320)@24fps'              -> '8K'
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    for pattern, label in RESOLUTION_PRIORITY:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return label

    return np.nan

def _extract_fps_values_after(text, start_pos):
    """
    Extract FPS values associated with the current resolution mention.

    Priority:
    1. Use FPS directly attached to the resolution with @...fps.
       Example: 4K@24/30/60fps -> [24, 30, 60]
    2. If no @...fps pattern exists, use nearby natural-language FPS.
       Example: 4K ở tốc độ 24 fps, 30 fps hoặc 60 fps -> [24, 30, 60]

    This avoids incorrectly taking slow-motion FPS from lower resolutions.
    """
    current_segment = text[start_pos:]

    # Stop before the next resolution mention
    next_resolution_positions = []

    for pattern, _ in RESOLUTION_PRIORITY:
        for match in re.finditer(pattern, current_segment, flags=re.IGNORECASE):
            if match.start() > 0:
                next_resolution_positions.append(match.start())

    if len(next_resolution_positions) > 0:
        end_pos = min(next_resolution_positions)
        current_segment = current_segment[:end_pos]

    # Keep only the part before common separators if possible.
    # This helps avoid cases like: 4K@30fps, 240fps @HD
    first_part = re.split(r"[,;]", current_segment)[0]

    # Case 1: FPS directly attached using @
    attached_match = re.search(
        r"@\s*([\d]+(?:\s*/\s*[\d]+)*)\s*f(?:ps|p)?",
        first_part,
        flags=re.IGNORECASE
    )

    if attached_match:
        return [
            int(x.strip())
            for x in attached_match.group(1).split("/")
        ]

    # Case 2: fallback for natural-language text near the resolution
    fps_matches = re.findall(
        r"(\d+)\s*f(?:ps|p)\b",
        first_part,
        flags=re.IGNORECASE
    )

    if len(fps_matches) == 0:
        return []

    return [int(x) for x in fps_matches]


def parse_fps_at_max_resolution(value):
    """
    Parse the best (maximum) FPS at the highest supported video resolution.

    Logic:
    1. Find the highest resolution in the text using RESOLUTION_PRIORITY order.
    2. Find all positions where that resolution appears in the text.
    3. For each position, extract all FPS values right after it (@X or @X/Y/Z).
    4. Return the highest FPS found across all occurrences.

    Examples:
    '4K@30fps, 1080p@120fps'           -> 30   (max FPS at 4K, not 1080p)
    '2160p@30/60fps'                   -> 60   (best FPS at 2160p)
    '4K@30fps, 4K@60fps, 1080p@120fps' -> 60  (4K appears twice, take max)
    '3240p@30fps, 2160p@30/60fps'      -> 30   (3240p is higher, take 30)
    'UHD 8K (7680 x 4320)@24fps'       -> 24
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    for pattern, label in RESOLUTION_PRIORITY:
        # Find all positions where this resolution appears
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))

        if not matches:
            continue

        # Collect all FPS values across all occurrences of this resolution
        all_fps = []
        for m in matches:
            fps_values = _extract_fps_values_after(text, m.start())
            all_fps.extend(fps_values)

        if all_fps:
            return max(all_fps)

        # Resolution found but no FPS value associated with it
        return np.nan

    return np.nan


print("Helper parsing functions for camera and video columns defined successfully.")

In [ ]:
# Apply camera parsing functions

# Parse front camera MP
df["front_camera_mp"] = df["front_camera_raw"].apply(parse_front_camera_mp)

# Parse rear main camera MP (first MP value = main lens)
df["rear_main_camera_mp"] = df["rear_camera_raw"].apply(parse_rear_main_camera_mp)

# Count number of rear camera lenses
df["rear_camera_count"] = df["rear_camera_raw"].apply(parse_rear_camera_count)

print("Camera columns parsed successfully.")

# Display raw columns and newly created columns side by side
display(
    df[
        [
            "product_name",
            "front_camera_raw",
            "front_camera_mp",
            "rear_camera_raw",
            "rear_main_camera_mp",
            "rear_camera_count"
        ]
    ].head(15)
)

In [ ]:
# Apply video parsing functions

# Parse maximum video resolution
df["max_video_resolution"] = df["video_raw"].apply(parse_max_video_resolution)

# Parse best FPS at maximum resolution
df["fps_at_max_resolution"] = df["video_raw"].apply(parse_fps_at_max_resolution)

print("Video columns parsed successfully.")

# Display raw columns and newly created columns side by side
display(
    df[
        [
            "product_name",
            "video_raw",
            "max_video_resolution",
            "fps_at_max_resolution"
        ]
    ].head(15)
)

In [ ]:
# Check camera and video parsing summary

camera_video_cols = [
    "front_camera_mp",
    "rear_main_camera_mp",
    "rear_camera_count",
    "max_video_resolution",
    "fps_at_max_resolution"
]

print("=" * 55)
print("BOOKMARK 5 — CAMERA AND VIDEO COLUMNS SUMMARY")
print("=" * 55)

for col in camera_video_cols:
    n_parsed  = df[col].notna().sum()
    n_missing = df[col].isna().sum()
    print(f"\n[ {col} ]")
    print(f"  Parsed  : {n_parsed}")
    print(f"  Missing : {n_missing}")
    if df[col].dtype != "object":
        print(f"  Min     : {df[col].min()}")
        print(f"  Max     : {df[col].max()}")
    else:
        print(f"  Unique values : {df[col].dropna().unique().tolist()}")

# 6. Handle Missing Values
Strategy:
- Group 1 — Mandatory columns: inspect rows with missing price or url.
- Group 2 — Categorical columns: fill missing with "Unknown".
- Group 3 — Numeric columns: keep NaN, do not impute with median.
- Group 4 — Clearly invalid numeric values: set to NaN.

In [ ]:
# Survey missing values in all parsed columns

parsed_cols = [
    "product_name_clean", "promotion_clean", "price_vnd",
    "ram_gb", "storage_gb", "screen_size_inch", "panel_clean",
    "refresh_rate_hz", "resolution_width", "resolution_height",
    "resolution_total_pixels", "battery_mah", "fast_charge_w",
    "weight_g", "brand", "os_family", "os_version", "ip_rating",
    "chip_name_clean", "front_camera_mp", "rear_main_camera_mp",
    "rear_camera_count", "max_video_resolution", "fps_at_max_resolution",
    "url"
]

missing_df = pd.DataFrame({
    "column": parsed_cols,
    "missing_count": [df[col].isna().sum() for col in parsed_cols],
    "missing_pct": [
        round(df[col].isna().mean() * 100, 2) for col in parsed_cols
    ]
}).sort_values("missing_count", ascending=False)

print("=" * 50)
print("MISSING VALUES SUMMARY — PARSED COLUMNS")
print("=" * 50)
display(missing_df[missing_df["missing_count"] > 0])

# Check mandatory columns separately only if they have missing values
mandatory_cols = ["product_name_clean", "price_vnd", "url"]

missing_mandatory = df[
    df[mandatory_cols].isna().any(axis=1)
]

print("\n" + "=" * 50)
print("MANDATORY COLUMNS CHECK")
print("=" * 50)

print(f"Rows with missing mandatory columns: {missing_mandatory.shape[0]}")

if not missing_mandatory.empty:
    display(
        missing_mandatory[
            [
                "product_name",
                "product_name_clean",
                "price_raw",
                "price_vnd",
                "url"
            ]
        ]
    )

In [ ]:
# Set clearly invalid numeric values to NaN

invalid_rules = {
    # Core specs
    "price_vnd":            lambda x: x <= 0,
    "screen_size_inch":     lambda x: x <= 0,
    "battery_mah":          lambda x: x <= 0,
    "weight_g":             lambda x: x <= 0,
    "refresh_rate_hz":      lambda x: x <= 0,
    "resolution_width":     lambda x: x <= 0,
    "resolution_height":    lambda x: x <= 0,
    # Memory — only catch exactly 0, not small valid values
    "ram_gb":               lambda x: x <= 0,
    "storage_gb":           lambda x: x <= 0,
    # Camera
    "front_camera_mp":      lambda x: x <= 0,
    "rear_main_camera_mp":  lambda x: x <= 0,
    "rear_camera_count":    lambda x: x <= 0,
    # Video
    "fps_at_max_resolution": lambda x: x <= 0,
    # Charging — negative only, 0 means no fast charge info
    "fast_charge_w":        lambda x: x < 0,
}

print("Validating numeric columns...")
print("-" * 45)

for col, rule in invalid_rules.items():
    n_invalid = df[col].apply(
        lambda x: rule(x) if pd.notna(x) else False
    ).sum()

    if n_invalid > 0:
        df.loc[
            df[col].apply(lambda x: rule(x) if pd.notna(x) else False),
            col
        ] = np.nan
        print(f"  {col}: {n_invalid} invalid value(s) set to NaN")
    else:
        print(f"  {col}: OK")

In [ ]:
# Fill missing categorical columns with "Unknown"

categorical_cols = [
    "panel_clean",
    "brand",
    "os_family",
    "ip_rating",
    "chip_name_clean",
    "max_video_resolution",
]

for col in categorical_cols:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        df[col] = df[col].fillna("Unknown")
        print(f"  {col}: {n_missing} missing value(s) filled with 'Unknown'")
    else:
        print(f"  {col}: no missing values")

In [ ]:
# Final check after missing value handling

print("=" * 55)
print("BOOKMARK 6 — MISSING VALUES AFTER HANDLING")
print("=" * 55)

still_missing = (
    pd.DataFrame({
        "column": parsed_cols,
        "missing_count": [df[col].isna().sum() for col in parsed_cols],
        "missing_pct": [
            round(df[col].isna().mean() * 100, 2) for col in parsed_cols
        ]
    })
    .sort_values("missing_count", ascending=False)
)

display(still_missing[still_missing["missing_count"] > 0])

print("\nCategorical columns — value counts after fill:")
for col in categorical_cols:
    print(f"\n[ {col} ]")
    print(df[col].value_counts().to_string())

# 7. Create Product Feature Columns

This bookmark creates derived feature columns from parsed columns.

Feature types created:
- Grouping   : price_segment, phone_type, panel_group, chip_brand
- Ranking    : video_resolution_rank
- Categorical: ip_status (1 = Certified / 0 = No rating or Unknown)



In [ ]:
# Create price_segment from price_vnd
#
# Segments based on Vietnamese smartphone market tiers:
#   budget   : under 5 million VND
#   mid      : 5 million to under 10 million VND
#   high     : 10 million to under 20 million VND
#   flagship : 20 million VND and above

def get_price_segment(price):
    if pd.isna(price):
        return "Unknown"
    if price < 5_000_000:
        return "budget"
    if price < 10_000_000:
        return "mid"
    if price < 20_000_000:
        return "high"
    return "flagship"

df["price_segment"] = df["price_vnd"].apply(get_price_segment)

print("price_segment created successfully.")
display(
    df[["product_name_clean", "price_vnd", "price_segment"]]
    .head(15)
)

print("\nDistribution:")
print(df["price_segment"].value_counts())

In [ ]:
# Create phone_type from screen_size_inch, ram_gb, storage_gb
#
# Feature phones are identified by any of:
#   - Screen size < 4 inches
#   - RAM < 0.5 GB (e.g. 48MB, 128MB)
#   - Storage < 1 GB
#
# These are heuristic rules based on dataset observation.
# Small ram_gb or storage_gb are NOT parse errors —
# they are real specs of feature phones (Masstel, Nokia, Viettel, etc.)
#
# Unknown: all three specs are missing, cannot determine type.

def get_phone_type(row):
    screen  = row["screen_size_inch"]
    ram     = row["ram_gb"]
    storage = row["storage_gb"]

    if pd.notna(screen) and screen < 4:
        return "feature_phone"
    if pd.notna(ram) and ram < 0.5:
        return "feature_phone"
    if pd.notna(storage) and storage < 1:
        return "feature_phone"
    if pd.isna(screen) and pd.isna(ram) and pd.isna(storage):
        return "Unknown"

    return "smartphone"

df["phone_type"] = df.apply(get_phone_type, axis=1)

print("phone_type created successfully.")
display(
    df[
        [
            "product_name_clean",
            "screen_size_inch",
            "ram_gb",
            "storage_gb",
            "phone_type"
        ]
    ].head(15)
)

print("\nDistribution:")
print(df["phone_type"].value_counts())

In [ ]:
# Create panel_group from panel_clean
#
# Groups many panel variants into main families:
#   AMOLED  : any AMOLED or OLED variant
#             (OLED and AMOLED are grouped together because
#              the difference is not meaningful for recommendation)
#   LCD     : any LCD variant
#   Other   : any panel type not matched above
#   Unknown : missing or unknown

def get_panel_group(value):
    if pd.isna(value) or value == "Unknown":
        return "Unknown"

    text = str(value).upper()

    if "AMOLED" in text or "OLED" in text or "RETINA" in text:
        return "AMOLED"
    if "LCD" in text or "IPS" in text:
        return "LCD"

    return "Other"

df["panel_group"] = df["panel_clean"].apply(get_panel_group)

print("panel_group created successfully.")
display(
    df[["product_name_clean", "panel_clean", "panel_group"]]
    .head(15)
)

print("\nDistribution:")
print(df["panel_group"].value_counts())

In [ ]:
# Create chip_brand from chip_name_clean
#
# Extracts chip manufacturer from chip name text.
# Order matters — more specific patterns are checked first.
#
# Apple chips are often written as "A18", "A19 Pro" without the word "Apple".
# Unisoc chips are often written as "T606", "T9300" without the word "Unisoc".
# Regex is used to catch these shorthand patterns.

def get_chip_brand(value):
    if pd.isna(value) or value == "Unknown":
        return "Unknown"

    text = str(value).upper()

    if "APPLE" in text or "BIONIC" in text or re.search(r"\bA\d{2}\b", text):
        return "Apple"
    if "SNAPDRAGON" in text:
        return "Snapdragon"
    if "DIMENSITY" in text or "HELIO" in text or "MEDIATEK" in text:
        return "MediaTek"
    if "EXYNOS" in text:
        return "Exynos"
    if "UNISOC" in text or "SPREADTRUM" in text or "UMS" in text \
            or "SC9863" in text or re.search(r"\bT\d{3,4}\b", text):
        return "Unisoc"
    if "ASR" in text:
        return "ASR"

    return "Other"

df["chip_brand"] = df["chip_name_clean"].apply(get_chip_brand)

print("chip_brand created successfully.")
display(
    df[["product_name_clean", "chip_name_clean", "chip_brand"]]
    .head(15)
)

print("\nDistribution:")
print(df["chip_brand"].value_counts())

In [ ]:
# Create video_resolution_rank from max_video_resolution
#
# Rank order (higher = better):
#   Unknown = 0
#   720p    = 1
#   1080p   = 2
#   2K      = 3
#   4K      = 4  (2160p)
#   3240p   = 5  (higher than 4K because 3240 > 2160 pixel rows)
#   8K      = 6  (4320p)

VIDEO_RANK_MAP = {
    "Unknown" : 0,
    "720p"    : 1,
    "1080p"   : 2,
    "2K"      : 3,
    "4K"      : 4,
    "3240p"   : 5,
    "8K"      : 6,
}

df["video_resolution_rank"] = (
    df["max_video_resolution"]
    .fillna("Unknown")
    .map(VIDEO_RANK_MAP)
    .fillna(0)
    .astype(int)
)

print("video_resolution_rank created successfully.")
display(
    df[
        [
            "product_name_clean",
            "max_video_resolution",
            "video_resolution_rank"
        ]
    ].head(15)
)

print("\nDistribution:")
print(df["video_resolution_rank"].value_counts().sort_index())

In [ ]:
# Create ip_status from ip_rating
#
# Binary encoding:
#   1 : product has a recognized IP certification (IP67, IP68, IPX4, etc.)
#   0 : no IP rating or specification not available
#
# Note:
#   Both "No rating" and "Unknown" are treated as 0.
#   Unknown means spec was not scraped — assumed not certified.

def get_ip_status(value):
    if pd.isna(value) or value == "Unknown":
        return 0
    if str(value).lower() == "none":
        return 0
    return 1

df["ip_status"] = df["ip_rating"].apply(get_ip_status)

print("ip_status created successfully.")
display(
    df[["product_name_clean", "ip_rating", "ip_status"]]
    .head(15)
)

print("\nDistribution:")
print(df["ip_status"].value_counts())

In [ ]:
# Final check for Bookmark 7

feature_cols = [
    "price_segment",
    "phone_type",
    "panel_group",
    "chip_brand",
    "video_resolution_rank",
    "ip_status",
]

print("=" * 55)
print("BOOKMARK 7 — FEATURE COLUMNS SUMMARY")
print("=" * 55)

for col in feature_cols:
    n_missing = df[col].isna().sum()
    print(f"\n[ {col} ]")
    print(f"  Missing : {n_missing}")
    print(df[col].value_counts().to_string())

# 8. Validate Preprocessed Dataset

This bookmark performs final validation checks before export.
No data is modified here — only checks and reports.

Checks:
- 8.1 Shape and expected columns
- 8.2 No missing in mandatory columns
- 8.3 Numeric columns within valid range
- 8.4 Categorical columns contain only allowed values
- 8.5 Final missing values summary

In [ ]:
# Check dataset shape and expected columns

EXPECTED_COLUMNS = [
    # Raw columns
    "product_name", "price_raw", "ram_raw", "storage_raw",
    "screen_size_raw", "panel_raw", "refresh_rate_raw", "resolution_raw",
    "battery_raw", "chip_raw", "brand_raw", "os_raw",
    "front_camera_raw", "rear_camera_raw", "video_raw",
    "charging_raw", "ip_raw", "weight_raw", "promotion_raw", "url",
    # Parsed columns — Bookmark 4
    "product_name_clean", "promotion_clean", "price_vnd",
    "ram_gb", "storage_gb", "screen_size_inch", "panel_clean",
    "refresh_rate_hz", "resolution_width", "resolution_height",
    "resolution_total_pixels", "battery_mah", "fast_charge_w",
    "weight_g", "brand", "os_family", "os_version",
    "ip_rating", "chip_name_clean",
    # Parsed columns — Bookmark 5
    "front_camera_mp", "rear_main_camera_mp", "rear_camera_count",
    "max_video_resolution", "fps_at_max_resolution",
    # Feature columns — Bookmark 7
    "price_segment", "phone_type", "panel_group",
    "chip_brand", "video_resolution_rank", "ip_status",
]

print("=" * 55)
print("CHECK 1 — SHAPE AND COLUMNS")
print("=" * 55)

print(f"\nDataset shape : {df.shape[0]} rows x {df.shape[1]} columns")

# Check for missing expected columns
missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]
extra_cols   = [col for col in df.columns if col not in EXPECTED_COLUMNS]

if missing_cols:
    print(f"\n❌ Missing expected columns ({len(missing_cols)}):")
    for col in missing_cols:
        print(f"   - {col}")
else:
    print(f"\n✅ All {len(EXPECTED_COLUMNS)} expected columns present.")

if extra_cols:
    print(f"\n⚠️  Extra columns not in expected list ({len(extra_cols)}):")
    for col in extra_cols:
        print(f"   - {col}")

In [ ]:
# Check no missing values in mandatory columns
#
# Mandatory columns are those that must be present for the
# recommendation system to function correctly.

MANDATORY_COLUMNS = [
    "product_name_clean",
    "price_vnd",
    "url",
    "brand",
    "os_family",
    "chip_name_clean",
    "price_segment",
    "phone_type",
    "panel_group",
    "chip_brand",
]

print("=" * 55)
print("CHECK 2 — MISSING IN MANDATORY COLUMNS")
print("=" * 55)

all_pass = True

for col in MANDATORY_COLUMNS:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        print(f"❌ {col}: {n_missing} missing value(s)")
        all_pass = False
    else:
        print(f"✅ {col}: OK")

if all_pass:
    print("\n✅ All mandatory columns have no missing values.")
else:
    print("\n❌ Some mandatory columns have missing values. Please review.")

In [ ]:
# Check numeric columns are within valid range
#
# Only rows with non-null values are checked.
# NaN values are allowed and handled separately in missing value summary.

NUMERIC_RANGE_CHECKS = {
    "price_vnd"       : (1_000, None),     # at least 1000 VND
    "screen_size_inch": (1.5,   10.0),
    "battery_mah"     : (100,   None),
    "refresh_rate_hz" : (30,    240),
    "ram_gb"          : (0,     None),     # > 0, no upper bound
    "storage_gb"      : (0,     None),     # > 0, no upper bound
    "fast_charge_w"   : (0,     None),     # >= 0
    "front_camera_mp" : (0,     None),
    "rear_main_camera_mp": (0,  None),
    "rear_camera_count"  : (1,  None),
    "video_resolution_rank": (0, 6),
}

print("=" * 55)
print("CHECK 3 — NUMERIC COLUMNS RANGE")
print("=" * 55)

all_pass = True

for col, (low, high) in NUMERIC_RANGE_CHECKS.items():
    series = df[col].dropna()

    if len(series) == 0:
        print(f"⚠️  {col}: no non-null values to check")
        continue

    violations = pd.Series([], dtype=float)

    if low is not None and high is not None:
        violations = series[(series < low) | (series > high)]
    elif low is not None:
        violations = series[series < low]
    elif high is not None:
        violations = series[series > high]

    if len(violations) > 0:
        print(f"❌ {col}: {len(violations)} value(s) out of range "
              f"[{low}, {high}] — min={series.min()}, max={series.max()}")
        all_pass = False
    else:
        print(f"✅ {col}: OK  (min={series.min():.2f}, max={series.max():.2f})")

if all_pass:
    print("\n✅ All numeric columns within valid range.")
else:
    print("\n❌ Some numeric columns have out-of-range values. Please review.")

In [ ]:
# Check categorical columns contain only allowed values

CATEGORICAL_ALLOWED = {
    "price_segment": {"budget", "mid", "high", "flagship", "Unknown"},
    "phone_type"   : {"smartphone", "feature_phone", "Unknown"},
    "panel_group"  : {"AMOLED", "LCD", "Unknown"},
    "ip_status"    : {0, 1},
    "os_family"    : {"Android", "iOS", "Other", "Unknown"},
    "chip_brand"   : {
        "Apple", "Snapdragon", "MediaTek",
        "Exynos", "Unisoc", "ASR", "Other", "Unknown"
    },
}

print("=" * 55)
print("CHECK 4 — CATEGORICAL COLUMNS ALLOWED VALUES")
print("=" * 55)

all_pass = True

for col, allowed in CATEGORICAL_ALLOWED.items():
    actual = set(df[col].dropna().unique())
    unexpected = actual - allowed

    if unexpected:
        print(f"❌ {col}: unexpected value(s): {unexpected}")
        all_pass = False
    else:
        print(f"✅ {col}: OK  {sorted(actual)}")

if all_pass:
    print("\n✅ All categorical columns contain only allowed values.")
else:
    print("\n❌ Some categorical columns have unexpected values. Please review.")

In [ ]:
# Bookmark 8.5 — Final missing values summary across all parsed columns

ALL_PARSED_COLS = [
    # Bookmark 4
    "product_name_clean", "promotion_clean", "price_vnd",
    "ram_gb", "storage_gb", "screen_size_inch", "panel_clean",
    "refresh_rate_hz", "resolution_width", "resolution_height",
    "resolution_total_pixels", "battery_mah", "fast_charge_w",
    "weight_g", "brand", "os_family", "os_version",
    "ip_rating", "chip_name_clean",
    # Bookmark 5
    "front_camera_mp", "rear_main_camera_mp", "rear_camera_count",
    "max_video_resolution", "fps_at_max_resolution",
    # Bookmark 7
    "price_segment", "phone_type", "panel_group",
    "chip_brand", "video_resolution_rank", "ip_status",
]

summary = pd.DataFrame({
    "column": ALL_PARSED_COLS,
    "missing_count": [df[col].isna().sum() for col in ALL_PARSED_COLS],
    "missing_pct"  : [
        round(df[col].isna().mean() * 100, 2) for col in ALL_PARSED_COLS
    ],
}).sort_values("missing_count", ascending=False)

print("=" * 55)
print("CHECK 5 — FINAL MISSING VALUES SUMMARY")
print("=" * 55)

display(summary)

print("\n" + "=" * 55)
print(f"Total columns checked : {len(ALL_PARSED_COLS)}")
print(f"Columns with missing  : {(summary['missing_count'] > 0).sum()}")
print(f"Columns fully complete: {(summary['missing_count'] == 0).sum()}")

# 9. Export Preprocessed Dataset

Export the final preprocessed dataset to CSV.


In [ ]:
# Export preprocessed dataset to CSV
from pathlib import Path

EXPORT_COLUMNS = [
    # --- Identifiers ---
    "product_name_clean",
    "brand",
    "url",

    # --- Price ---
    "price_vnd",
    "price_segment",
    "promotion_clean",

    # --- Phone type ---
    "phone_type",
    "os_family",
    "os_version",
    "chip_name_clean",
    "chip_brand",

    # --- Display ---
    "screen_size_inch",
    "panel_clean",
    "panel_group",
    "refresh_rate_hz",
    "resolution_width",
    "resolution_height",
    "resolution_total_pixels",

    # --- Memory ---
    "ram_gb",
    "storage_gb",

    # --- Battery & charging ---
    "battery_mah",
    "fast_charge_w",

    # --- Camera ---
    "front_camera_mp",
    "rear_main_camera_mp",
    "rear_camera_count",

    # --- Video ---
    "max_video_resolution",
    "fps_at_max_resolution",
    "video_resolution_rank",

    # --- Build ---
    "weight_g",
    "ip_status",
]

# Rename columns for cleaner output
# _clean suffix was used internally to distinguish from raw columns.
# In the final export, raw columns are excluded so the suffix is unnecessary.
RENAME_FOR_EXPORT = {
    "product_name_clean" : "product_name",
    "promotion_clean"    : "promotion",
    "panel_clean"        : "panel",
    "chip_name_clean"    : "chip_name",
}

df_export = (
    df[EXPORT_COLUMNS]
    .copy()
    .rename(columns=RENAME_FOR_EXPORT)
)

output_path = Path("../../data/processed/phone_specs_preprocessed.csv")

# Create processed folder if it does not exist
output_path.parent.mkdir(parents=True, exist_ok=True)

df_export.to_csv(output_path, index=False, encoding="utf-8-sig")

print("=" * 55)
print("BOOKMARK 9 — EXPORT SUMMARY")
print("=" * 55)
print(f"\nExported to   : {output_path}")
print(f"Rows          : {df_export.shape[0]}")
print(f"Columns       : {df_export.shape[1]}")
print(f"\nColumn list:")
for i, col in enumerate(df_export.columns, start=1):
    print(f"  {i:02d}. {col}")
print("\n" + "=" * 55)
print("Preprocessing pipeline completed successfully.")
print("=" * 55)